<a href="https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is Structured Content Archetype Clustering, so the method is KMeans clustering, from the menu's clustering option. Clustering fits because there's no label to predict — the goal is to discover natural groupings of content by behavior (impressions, clicks, position, engagement) so content teams can treat groups differently. I'll pick k using a silhouette sweep, and compare the resulting clusters against my Week-4 rule-based baseline (fix_ctr vs monitor) to see whether clustering surfaces the same at-risk content, different content, or a genuinely richer picture.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Clustering is unsupervised, so there's no train/test accuracy to protect against overfitting in the classic sense — but I still need an honest validation design to check that clusters aren't just noise or a single dominant client's pattern. I use a grouped split by client: fit KMeans on a subset of clients, then check that cluster assignments on held-out clients look similarly structured (similar per-cluster sizes and feature means), rather than the clustering only working because of one client's idiosyncratic scale. This matters because gsc_impressions and similar counts vary hugely by client size, so a random row-level split could leak client identity across train/test in a way that hides poor generalization.

In [15]:
import pandas as pd
import numpy as np
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

fact_cols = ["report_date","client_hash_id","content_hash_id","gsc_data_available","ga4_data_available",
             "gsc_impressions","gsc_clicks","gsc_avg_position","ga4_pageviews","ga4_sessions",
             "ga4_engaged_sessions","ga4_total_engagement_sec"]
panel_daily = pd.read_parquet(
    f"{HF_BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    columns=fact_cols, storage_options={"token": HF_TOKEN}
)
dim_cols = ["client_hash_id","content_hash_id","content_type"]
dim_content = pd.read_parquet(f"{HF_BASE}/dim_content.parquet", columns=dim_cols, storage_options={"token": HF_TOKEN})

for col in ["client_hash_id","content_hash_id"]:
    panel_daily[col] = panel_daily[col].astype("category")
    dim_content[col] = dim_content[col].astype("category")

dim_content_clean = dim_content.drop_duplicates(subset=["client_hash_id","content_hash_id"], keep="first")
panel_daily = panel_daily.merge(dim_content_clean, on=["client_hash_id","content_hash_id"], how="left")

content_level = panel_daily.groupby(["client_hash_id","content_hash_id"], observed=True).agg(
    gsc_impressions=("gsc_impressions","sum"),
    gsc_clicks=("gsc_clicks","sum"),
    gsc_avg_position=("gsc_avg_position","mean"),
    ga4_engaged_sessions=("ga4_engaged_sessions","sum"),
    content_type=("content_type","first"),
).reset_index()

content_level["ctr"] = content_level["gsc_clicks"] / content_level["gsc_impressions"].replace(0, np.nan)
content_level["engagement_rate"] = content_level["ga4_engaged_sessions"] / content_level["gsc_clicks"].replace(0, np.nan)

print("Shape:", content_level.shape)

Shape: (331437, 9)


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

groups = content_level["client_hash_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(content_level, groups=groups))

train_df = content_level.iloc[train_idx].copy()
test_df = content_level.iloc[test_idx].copy()

print("Train clients:", train_df["client_hash_id"].nunique(), "| Train rows:", len(train_df))
print("Test clients:", test_df["client_hash_id"].nunique(), "| Test rows:", len(test_df))
print("Any client overlap?", bool(set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])))

Train clients: 38 | Train rows: 281614
Test clients: 17 | Test rows: 49823
Any client overlap? False


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

feature_cols = ["gsc_impressions","gsc_clicks","gsc_avg_position","ga4_engaged_sessions"]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols].fillna(0))
X_test = scaler.transform(test_df[feature_cols].fillna(0))

# k-sweep on train
rng = np.random.default_rng(42)
sweep = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train)
    idx = rng.choice(len(X_train), size=min(20000, len(X_train)), replace=False)
    sweep[k] = silhouette_score(X_train[idx], km.labels_[idx])
print("Silhouette by k:", sweep)

best_k = max(sweep, key=sweep.get)
print("Chosen k:", best_k)

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X_train)
train_df["cluster"] = km_final.labels_
test_df["cluster"] = km_final.predict(X_test)

idx_test = rng.choice(len(X_test), size=min(20000, len(X_test)), replace=False)
test_silhouette = silhouette_score(X_test[idx_test], test_df["cluster"].values[idx_test])
print("Held-out (test) silhouette:", round(test_silhouette, 3))

Silhouette by k: {2: np.float64(0.8889355712706947), 3: np.float64(0.7197478986909323), 4: np.float64(0.7348320028199199), 5: np.float64(0.7362882785205663), 6: np.float64(0.740764004250591), 7: np.float64(0.7472310733266447)}
Chosen k: 2
Held-out (test) silhouette: 0.911


In [18]:
# Baseline comparison — rebuild w04's rule on the SAME data/split
CTR_THRESHOLD = 0.005

def score_row(row):
    if row["gsc_impressions"] == 0:
        return "monitor"
    if pd.notna(row["ctr"]) and row["gsc_avg_position"] <= 10 and row["ctr"] < CTR_THRESHOLD:
        return "fix_ctr"
    return "monitor"

test_df["baseline_action"] = test_df.apply(score_row, axis=1)

# Compare: does the model's clustering agree with / add to the baseline's binary call?
comparison = test_df.groupby(["cluster","baseline_action"], observed=True).size().unstack(fill_value=0)
print(comparison)

print("\nModel vs baseline summary table:")
summary = pd.DataFrame({
    "method": ["Baseline (rule)", "Model (KMeans)"],
    "metric_name": ["fix_ctr rate", "silhouette (held-out)"],
    "value": [
        round((test_df["baseline_action"] == "fix_ctr").mean(), 4),
        round(test_silhouette, 4)
    ]
})
print(summary)

baseline_action  fix_ctr  monitor
cluster                          
0                  12535    36608
1                    200      480

Model vs baseline summary table:
            method            metric_name   value
0  Baseline (rule)           fix_ctr rate  0.2556
1   Model (KMeans)  silhouette (held-out)  0.9105


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Where it's wrong: with k=2, the model can't distinguish why Cluster 0 content underperforms — a page with zero impressions and a page with 500 impressions but the wrong search intent look statistically similar here but need completely different fixes. A higher k (or a second modeling pass restricted to Cluster 0) would likely be needed to find that nuance — worth flagging as a limitation rather than a k=2 model claiming to fully capture archetype diversity. Also, the wide within-cluster variance (n=278,705 is very heterogeneous) means Cluster 0 is really "not Cluster 1" more than a coherent archetype on its own.

What the model leans on: the split is almost entirely driven by scale (gsc_impressions, gsc_clicks) — Cluster 1's impressions are ~44x higher than Cluster 0's. Position and engagement differ too, but far less dramatically in relative terms, so KMeans is mostly separating "big" content from "everything else," not finding a nuanced multi-axis archetype structure.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Interpret clusters: what does each one look like?
cluster_profile = train_df.groupby("cluster", observed=True)[feature_cols].mean()
cluster_profile["n"] = train_df.groupby("cluster", observed=True).size()
print(cluster_profile)

         gsc_impressions  gsc_clicks  gsc_avg_position  ga4_engaged_sessions  \
cluster                                                                        
0             577.225683    1.436583         16.312187              0.043501   
1           25652.746992   93.764524         11.556108              3.220694   

              n  
cluster          
0        278705  
1          2909  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.